<a href="https://colab.research.google.com/github/irullah/comic-yolo/blob/main/Ruru_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 🛠️ Langkah 1: Instalasi Library & Dependensi
Sel ini menginstal pustaka **Ultralytics** (mesin deteksi objek YOLOv8) dan **IOPaint** (mesin *deep learning inpainting* yang dioptimalkan untuk GPU).

> **Catatan Penting:** Jika setelah instalasi muncul tombol atau peringatan **"Restart runtime"**, klik tombol restart tersebut **1 kali**, lalu lanjutkan langsung ke Langkah 2 (tidak perlu menjalankan ulang sel ini).

### 📂 Langkah 2: Menghubungkan Google Drive
Menghubungkan Google Drive ke sesi Colab untuk memuat file model kustom YOLO (`best.pt`) yang tersimpan di Drive Anda.

### 🚀 Langkah 3: Eksekusi Batch Cleaning (YOLO + Anime-LaMa)
Alur kerja otomatis untuk membersihkan teks pada gambar komik/webtoon dari file ZIP:
* **Deteksi Teks:** YOLOv8 memindai balon kata menggunakan teknik *sliding window* dengan koordinat poligon presisi.
* **Model AI `anime-lama`:** Menggunakan varian model LaMa yang dilatih khusus untuk memulihkan tekstur 2D, *screentone*, dan garis komik (*lineart*).
* **Fitur HDStrategy.CROP:** IOPaint otomatis memotong (*crop*) hanya pada area sekitar balon kata, menghapus teksnya, lalu menempelkannya kembali. Menjaga gambar tetap tajam 100% dan mencegah GPU kehabisan VRAM (*Out of Memory*).
* **Otomasi File:** Mendeteksi file ZIP di sesi, mengekstrak, memproses seluruh halaman, lalu mengemas hasilnya ke file `C.zip` yang terunduh otomatis.

### 🎨 Langkah 4 (Opsional): Antarmuka Editor Visual WebUI
Jalankan sel ini hanya jika ingin menghapus efek suara (*SFX*) rumit atau memperbaiki detail gambar secara manual lewat kuas interaktif di browser melalui jalur *Cloudflare Tunnel*.

In [ ]:
%pip install -q ultralytics iopaint --no-warn-conflicts

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import cv2
import torch
import numpy as np
import zipfile
import shutil
from ultralytics import YOLO
from iopaint.model_manager import ModelManager
from iopaint.schema import InpaintRequest, HDStrategy
from google.colab import files

# Fungsi pembantu membaca/menyimpan nama file dengan karakter khusus (Unicode/Jepang/Korea)
def read_image_utf8(path):
    img_array = np.fromfile(path, np.uint8)
    return cv2.imdecode(img_array, cv2.IMREAD_COLOR)

def write_image_utf8(path, img):
    ext = os.path.splitext(path)[1]
    is_success, im_buf_arr = cv2.imencode(ext, img)
    if is_success:
        im_buf_arr.tofile(path)

# =====================================================================
# [LANGKAH 1]: PERSIAPAN AI (IOPaint Anime-LaMa & Custom YOLO)
# =====================================================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[1/5] Menyiapkan Mesin IOPaint (anime-lama) pada {device}...")

# Menggunakan 'anime-lama' (khusus komik/manga/webtoon).
# Bisa diganti menjadi 'lama' jika ingin menggunakan model LaMa standar.
iopaint_model = ModelManager(name="anime-lama", device=device)

# Konfigurasi HD Strategy CROP khas IOPaint agar hemat VRAM & tajam
iopaint_config = InpaintRequest(
    hd_strategy=HDStrategy.CROP,
    hd_strategy_crop_trigger_size=800, # Aktifkan crop otomatis jika dimensi > 800px
    hd_strategy_crop_margin=128        # Margin konteks di sekitar balon teks (piksel)
)

model_path = "/content/drive/MyDrive/Colab Notebooks/Comic Cleaner /best.pt"

print("[2/5] Menyalakan Radar Deteksi Teks (YOLOv8 Polygon)...")
text_detector = YOLO(model_path)

# =====================================================================
# [LANGKAH 2]: DETEKSI & PILIH FILE ZIP DARI SESSION COLAB
# =====================================================================
input_dir = "input_komik"
output_dir = "output_komik"

# Cari semua file .zip di direktori kerja (kecuali file output 'C.zip' sebelumnya)
zip_candidates = [f for f in os.listdir('.') if f.lower().endswith('.zip') and not f.startswith('C')]

if not zip_candidates:
    print("❌ ERROR: Tidak ditemukan file .zip di panel Files Colab!")
    print("💡 Silakan upload/tarik file .zip ke panel Files di sebelah kiri, lalu jalankan kembali cell ini.")
else:
    # Logika pemilihan file ZIP
    if len(zip_candidates) == 1:
        zip_path = zip_candidates[0]
        print(f"\n[3/5] Menggunakan file ZIP yang terdeteksi: {zip_path}")
    else:
        print(f"\n[3/5] Ditemukan {len(zip_candidates)} file ZIP di session:")
        for idx_zip, zf_name in enumerate(zip_candidates, 1):
            print(f"  [{idx_zip}] {zf_name}")

        while True:
            try:
                pilihan = int(input(f"Pilih nomor file yang ingin diproses (1-{len(zip_candidates)}): "))
                if 1 <= pilihan <= len(zip_candidates):
                    zip_path = zip_candidates[pilihan - 1]
                    print(f"-> Dipilih: {zip_path}")
                    break
                print("⚠️ Nomor tidak valid, coba lagi.")
            except ValueError:
                print("⚠️ Masukkan angka yang sesuai.")

    # Bersihkan folder dari sisa proses sebelumnya
    if os.path.exists(input_dir): shutil.rmtree(input_dir)
    if os.path.exists(output_dir): shutil.rmtree(output_dir)

    os.makedirs(input_dir, exist_ok=True)
    os.makedirs(output_dir, exist_ok=True)

    print(f"\nMengekstrak {zip_path}...")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall(input_dir)

    valid_extensions = ('.jpg', '.jpeg', '.png', '.webp')
    image_files = []
    for root, dirs, files_in_dir in os.walk(input_dir):
        for file in files_in_dir:
            if file.lower().endswith(valid_extensions):
                image_files.append(os.path.join(root, file))

    # Urutkan nama file agar rapi saat diproses
    image_files.sort()

    total_images = len(image_files)
    if total_images == 0:
        print("❌ ERROR: Tidak ditemukan file gambar (.jpg/.jpeg/.png/.webp) di dalam file ZIP tersebut!")
    else:
        # =====================================================================
        # [LANGKAH 3 & 4]: BATCH PROCESSING DENGAN IOPAINT
        # =====================================================================
        print(f"\n[4/5] Memulai Batch Processing (IOPaint Mode) untuk {total_images} gambar...")

        for idx, img_path in enumerate(image_files, 1):
            rel_path = os.path.relpath(img_path, input_dir)
            save_path = os.path.join(output_dir, rel_path)
            os.makedirs(os.path.dirname(save_path), exist_ok=True)

            print(f"---> Memproses [{idx}/{total_images}]: {rel_path}...")

            img = read_image_utf8(img_path)
            if img is None:
                print(f"⚠️ Gagal membaca {rel_path}, dilewati.")
                continue

            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            mask_teks = np.zeros(img.shape[:2], dtype="uint8")

            # --- DETEKSI SLIDING WINDOW (YOLO) ---
            H, W = img_rgb.shape[:2]
            slice_h = W
            overlap = int(slice_h * 0.1)
            step = slice_h - overlap

            for y in range(0, H, step):
                y1 = y
                y2 = min(y + slice_h, H)
                if y2 - y1 < 50:
                    break

                crop_img = img_rgb[y1:y2, 0:W]
                results = text_detector.predict(crop_img, conf=0.10, verbose=False)[0]

                if results.masks is not None:
                    polygons = results.masks.xy
                    for poly in polygons:
                        poly_pts = np.array(poly, dtype=np.int32)
                        poly_pts[:, 1] += y1
                        cv2.fillPoly(mask_teks, [poly_pts], 255)

            # Jika gambar tidak memiliki teks sama sekali, langsung simpan gambar asli
            if not np.any(mask_teks > 0):
                write_image_utf8(save_path, img)
                print(f"⏩ {rel_path} tidak ada teks, langsung disimpan!")
                continue

            # --- DILATION DINAMIS (Sudah diperbaiki dari typo) ---
            k_size = max(3, int(img.shape[1] * 0.010))
            if k_size % 2 == 0:
                k_size += 1
            kernel = np.ones((k_size, k_size), np.uint8)
            mask_final = cv2.dilate(mask_teks, kernel, iterations=1)

            # --- IOPAINT INPAINTING (CROP STRATEGY) ---
            # Catatan: IOPaint menerima input RGB dan langsung menghasilkan output dalam format BGR
            hasil_bgr = iopaint_model(img_rgb, mask_final, iopaint_config)

            # Simpan hasil akhir
            write_image_utf8(save_path, hasil_bgr)

            print(f"✅ {rel_path} selesai dibersihkan!")

        # =====================================================================
        # [LANGKAH 5]: ZIP HASIL & DOWNLOAD OTOMATIS
        # =====================================================================
        print("\n[5/5] Membungkus semua gambar yang sudah bersih ke file ZIP...")
        output_zip_name = "C"
        shutil.make_archive(output_zip_name, 'zip', output_dir)

        print("Memicu download otomatis ke perangkat Anda...")
        files.download(f"{output_zip_name}.zip")

        print("\n🎉 === SEMUA PROSES SELESAI! ===")

In [ ]:
# Opsional: Menyalakan WebUI IOPaint dengan publik link (Cloudflared)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

import subprocess
import time
import re

# Jalankan server IOPaint di latar belakang pada port 8080
iopaint_proc = subprocess.Popen(
    ["iopaint", "start", "--model=anime-lama", "--device=cuda", "--port=8080"],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

time.sleep(5)

# Buat tunnel agar bisa dibuka di browser luar Colab
tunnel_proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:8080"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True
)

print("⏳ Sedang membuat link WebUI IOPaint...")
for line in tunnel_proc.stdout:
    match = re.search(r"https://[-a-zA-Z0-9]+\.trycloudflare\.com", line)
    if match:
        print(f"\n🎨 Klik link ini untuk membuka Editor WebUI IOPaint:\n👉 {match.group(0)}")
        break